# Batch-evaluate the FoldBench test set using AF3-predicted structures, save the results, and compute performance metrics.

# Import environments

In [1]:
import os
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn.metrics import roc_auc_score
from tqdm.auto import tqdm
from torch_geometric.loader import DataLoader as GeoDataLoader

from utils.dataset import RNAProteinMoleculeDataset
from utils.utils_with_inter_edge_feat import pad_conmap_truth_lbl, rigid_transform_perturb_batch
from utils.contact_metrics import precision_topk as _precision_topk
from utils.Predictor_inter import Predictor, read_batch
from models.ZHMolContact_with_refine import ZHMolContact

os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"  # Configure deterministic cuBLAS operations

In [2]:
evaluate_epoch = 1
num_classes = 37
seed = 2024
data_path = './Data/'

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
torch.use_deterministic_algorithms(
    True,
    warn_only=True
)

# Import function

In [3]:
def parse_fasta(file_path):
    """
    Parse a FASTA file and return its header and sequence.
    """
    with open(file_path, 'r') as f:
        lines = f.readlines()

    header = lines[0].strip()  # FASTA header
    sequence = ''.join(line.strip() for line in lines[1:])  # Sequence

    return header, sequence


def logits_to_contact_score(
    logits: torch.Tensor,
    num_classes: int,
    contact_class_cutoff: int = 16
) -> torch.Tensor:
    """
    Convert multi-class distance logits into contact probability scores.

    Args:
        logits: Input tensor of shape [B, C, H, W].
        num_classes: Total number of distance classes.
        contact_class_cutoff: Number of classes considered as contacts
            when num_classes == 37.

    Returns:
        torch.Tensor: Contact probability scores of shape [B, H, W],
            where higher values indicate higher contact probabilities.
    """
    # Move the channel dimension to the last axis: [B, C, H, W] -> [B, H, W, C]
    probs = torch.softmax(logits.permute(0, 2, 3, 1), dim=-1)

    if num_classes == 37:
        # Sum the probabilities of the first contact_class_cutoff classes
        score = probs[..., :contact_class_cutoff].sum(-1)
    else:
        # Use the probability of the first class by default
        score = probs[..., :1].sum(-1)

    return score  # [B, H, W]


def compute_precision_topk(score, target, topk_list, topk_str_list):
    """Compute precision for each named K, excluding padding."""
    if len(topk_list) != len(topk_str_list):
        raise ValueError('Each K value must have a metric name')
    values = _precision_topk(score, target, topk_list)
    return {name: values[k] for k, name in zip(topk_list, topk_str_list)}


# Import Data

In [4]:
# ==================== Load Test Dataset ====================
test_file = f"{data_path}/FoldBench_set.csv"
test_df = pd.read_csv(test_file).reset_index(drop=True)

# Complex identifiers
pdbcodes = test_df["PDB_code"].tolist()

# ==================== Dataset Summary ====================
print(f"Test dataset size: {len(test_df)}")

Test dataset size: 69


In [5]:
RNA_path = os.path.join(f'{data_path}','FoldBench_set_RNA_fea.pt')
print('Loading RNA Graph data...')
RNA_dict = torch.load(RNA_path, map_location='cpu')

protein_path = os.path.join(f'{data_path}','FoldBench_set_protein_fea.pt')
print('Loading Protein Graph data...')
prot_dict = torch.load(protein_path, map_location='cpu')

RP_path = os.path.join(f'{data_path}', 'FoldBench_set_label.pt')
print('Loading contact map data...')
RP_conmap_truth_dict = torch.load(RP_path, map_location='cpu')

prior_file = f'{data_path}/FoldBench_prior_fea.pt'
print('Loading prior data...')
prior_dict = torch.load(prior_file, map_location='cpu')

Loading RNA Graph data...
Loading Protein Graph data...
Loading contact map data...
Loading prior data...


In [6]:
test_dataset = RNAProteinMoleculeDataset(test_df, RNA_dict, prot_dict, RP_conmap_truth_obj=RP_conmap_truth_dict, prior_dict=prior_dict, device=device)
print(test_dataset)

print("Loading data by DataLoader...")

test_shuffle = False
test_sampler = None
test_loader = GeoDataLoader(test_dataset, batch_size=1, shuffle=test_shuffle,
                        sampler=test_sampler, follow_batch=['RNA_node_aa', 'prot_node_aa'])

new_test_loader = []
for data in test_loader:
    # ii += 1
    # if ii < 10:
    print(data)
    padded_data = pad_conmap_truth_lbl(data)
    perturb_coordinate_data = rigid_transform_perturb_batch(padded_data, seed=seed, oprate_trans=False)
    new_test_loader.append(perturb_coordinate_data)
    
print(new_test_loader)


RNAProteinMoleculeDataset(69)
Loading data by DataLoader...
MultiGraphDataBatch(RNA_node_aa=[109, 21], RNA_node_aa_batch=[109], RNA_node_aa_ptr=[2], RNA_node_evo=[109, 640], RNA_node_pos=[109, 1], RNA_seq=[1], RNA_edge_index=[2, 707], RNA_edge_weight=[707], RNA_num_nodes=[1], RNA_xyz_coord=[109, 3], RNA_xyz_coord_P=[109, 3], RNA_xyz_coord_C4=[109, 3], RNA_xyz_coord_N=[109, 3], prot_node_aa=[1198, 33], prot_node_aa_batch=[1198], prot_node_aa_ptr=[2], prot_node_evo=[1198, 1280], prot_node_pos=[1198, 1], prot_seq=[1], prot_edge_index=[2, 11352], prot_edge_weight=[11352], prot_num_nodes=[1], prot_xyz_coord=[1198, 3], prot_xyz_coord_CA=[1198, 3], prot_xyz_coord_C=[1198, 3], prot_xyz_coord_N=[1198, 3], protein_domain_mask=[1198], conmap_truth=[1], conmap_truth_strict=[1], conmap_truth_origin=[1], conmap_truth_resize=[32, 512], conmap_truth_lbl=[1], RNA_site_truth=[1], prot_site_truth=[1], RNA_pred_site=[1], prot_pred_site=[1], elec_edge_feat=[1], hbond_edge_feat=[1], pi_pi_edge_feat=[1], sas

# Load trained model

In [7]:
# ==================== Model Configuration ====================
model_dict_path = (
    "trained_model/model.pth"
)

dropout = 0.05
heads = 4
H = 48
n_ls = 2
n_tri_ls = 2

# ==================== Initialize Model ====================
zhmolcontact = ZHMolContact(
    dropout=dropout,
    node_feat_dropout=dropout,
    evo_feat_dropout=dropout,
    heads=heads,
    hidden_channels=H,
    num_layers=n_ls,
    num_triangle_layers=n_tri_ls
)

# ==================== Load Pretrained Weights ====================
checkpoint = torch.load(model_dict_path, map_location='cpu')

state_dict = (
    checkpoint["state_dict"]
    if isinstance(checkpoint, dict) and "state_dict" in checkpoint
    else checkpoint
)

zhmolcontact.load_state_dict(state_dict)

# ==================== Prepare Model for Inference ====================
zhmolcontact = zhmolcontact.to(device)
torch.use_deterministic_algorithms(
    False
)
zhmolcontact.eval()

# ==================== Initialize Predictor ====================
predictor = Predictor(
    model=zhmolcontact,
    device=device
)

print(f"Model loaded successfully: {model_dict_path}")
print(f"Inference device: {device}")

Model loaded successfully: trained_model/model.pth
Inference device: cuda:0


# Evaluate

In [8]:
# ==================== Evaluation Configuration ====================
result_dir = "result/FoldBench_set/"

pred_dir = os.path.join(result_dir, "contact_probs")
LFT_pred_dir = os.path.join(result_dir, "contact_probs_LFT")
pred_logits_dir = os.path.join(result_dir, "contact_probs_logits")

prec_save_path = os.path.join(
    result_dir, "ZHMolContact_precision_results.csv"
)

# Create output directories
for directory in [pred_dir, LFT_pred_dir, pred_logits_dir]:
    os.makedirs(directory, exist_ok=True)

# Initialize evaluation results
all_res_p = []

topk_names = [
    "top1", "top2", "top3", "top4", "top5",
    "top6", "top7", "top8", "top9", "top10",
    "top L/2", "top L"
]

# ==================== Model Evaluation ====================
zhmolcontact.eval()

for data in tqdm(new_test_loader, desc="Evaluating FoldBench"):

    # ==================== Prepare Input ====================
    data = pad_conmap_truth_lbl(data)
    data = rigid_transform_perturb_batch(data, seed=seed, oprate_trans=False)
    data = data.to(device)
    test_id = data.pdbname[0]

    print(f"\n{'=' * 80}")
    print(f"Evaluating: {test_id}")

    # ==================== Model Prediction ====================
    with torch.no_grad():
        final_logits = predictor.predict(data=data)

        # Convert distance logits to contact probabilities
        final_probs = logits_to_contact_score(
            final_logits,
            num_classes=37,
            contact_class_cutoff=16
        )

    # Remove batch dimension: [1, H, W] -> [H, W]
    final_probs_2d = final_probs.squeeze(0)

    # Get RNA sequence length
    L = final_probs_2d.shape[0]

    # ==================== Save Contact Probabilities ====================
    final_probs_np = final_probs_2d.detach().cpu().numpy()

    # Save contact probability matrix
    pred_csv_path = os.path.join(pred_dir, f"{test_id}.csv")

    pd.DataFrame(final_probs_np).to_csv(
        pred_csv_path,
        index=False,
        header=False
    )

    # Save contact probabilities in long format
    final_probs_df = (
        pd.DataFrame(final_probs_np)
        .stack()
        .reset_index()
    )

    final_probs_df.columns = ["row", "col", "value"]

    LFT_pred_csv_path = os.path.join(
        LFT_pred_dir, f"{test_id}.csv"
    )

    final_probs_df.to_csv(
        LFT_pred_csv_path,
        index=False
    )

    # ==================== Save Distance Probabilities ====================
    # Convert logits to probabilities: [B, C, H, W] -> [B, H, W, C]
    distance_probs = torch.softmax(
        final_logits.permute(0, 2, 3, 1),
        dim=-1
    ).squeeze(0)

    # Save distance probability distributions
    pred_logits_path = os.path.join(
        pred_logits_dir, f"{test_id}.pt"
    )

    torch.save(
        distance_probs.detach().cpu(),
        pred_logits_path
    )

    # ==================== Prepare Ground-Truth Labels ====================
    lbl_target = data.conmap_truth_lbl_pad  # [B, H, W]
    res_lbl = lbl_target[0]

    # Convert distance labels to binary contact labels
    # Contact: distance class < 16
    # Non-contact: distance class >= 16
    # Padding: -100
    res_true_binary = torch.full_like(
        res_lbl,
        -100,
        dtype=torch.float32
    )

    valid_mask = res_lbl != -100

    res_true_binary[valid_mask] = (
        res_lbl[valid_mask] < 16
    ).float()

    # ==================== Compute Top-K Precision ====================
    topk_list = [
        1, 2, 3, 4, 5,
        6, 7, 8, 9, 10,
        L // 2, L
    ]

    res_p = compute_precision_topk(
        final_probs_2d,
        res_true_binary,
        topk_list,
        topk_names
    )

    res_p["pdbname"] = test_id
    all_res_p.append(res_p)

    print(f"Prediction shape: {tuple(final_probs_2d.shape)}")
    print(f"Ground-truth shape: {tuple(res_true_binary.shape)}")
    print(f"Top-K precision: {res_p}")

# ==================== Save Evaluation Results ====================
prec_df = pd.DataFrame(all_res_p)

# Reorder columns
prec_df = prec_df[["pdbname"] + topk_names]

# Save evaluation metrics
prec_df.to_csv(prec_save_path, index=False)

print(f"\n{'=' * 80}")
print(f"Evaluation completed: {len(prec_df)} complexes")
print(f"Precision results saved to: {prec_save_path}")
print(f"Contact probabilities saved to: {pred_dir}")
print(f"Long-format predictions saved to: {LFT_pred_dir}")
print(f"Distance probabilities saved to: {pred_logits_dir}")

Evaluating FoldBench:   0%|          | 0/69 [00:00<?, ?it/s]


Evaluating: 8IYQ_O_A
Prediction shape: (109, 1198)
Ground-truth shape: (109, 1198)
Top-K precision: {'top1': 0.0, 'top2': 0.0, 'top3': 0.0, 'top4': 0.25, 'top5': 0.20000000298023224, 'top6': 0.1666666716337204, 'top7': 0.1428571492433548, 'top8': 0.125, 'top9': 0.1111111119389534, 'top10': 0.10000000149011612, 'top L/2': 0.24074074625968933, 'top L': 0.23853211104869843, 'pdbname': '8IYQ_O_A'}

Evaluating: 8JOZ_B_A
Prediction shape: (88, 257)
Ground-truth shape: (88, 257)
Top-K precision: {'top1': 1.0, 'top2': 1.0, 'top3': 1.0, 'top4': 1.0, 'top5': 1.0, 'top6': 1.0, 'top7': 1.0, 'top8': 1.0, 'top9': 1.0, 'top10': 0.8999999761581421, 'top L/2': 0.6363636255264282, 'top L': 0.5340909361839294, 'pdbname': '8JOZ_B_A'}

Evaluating: 9C69_C_A
Prediction shape: (4, 164)
Ground-truth shape: (4, 164)
Top-K precision: {'top1': 1.0, 'top2': 1.0, 'top3': 0.6666666865348816, 'top4': 0.75, 'top5': 0.6000000238418579, 'top6': 0.6666666865348816, 'top7': 0.7142857313156128, 'top8': 0.625, 'top9': 0.66

In [9]:
# ==================== Summary Statistics ====================
# Compute mean Top-K precision across all test complexes

print(f"\n{'=' * 60}")
print(f"{'ZHMolContact | Top-K precision Evaluation Summary':^60}")
print(f"{'=' * 60}")
mean_precision = prec_df[topk_names].mean()

# Create summary table
summary_df = pd.DataFrame({
    "Top-K": mean_precision.index,
    "Mean Precision": mean_precision.values
})

# Display formatted summary table
print(f"ZHMolContact | Evaluation Summary (N = {len(prec_df)})")

display(
    summary_df.style
    .format({
        "Mean Precision": "{:.4f}"
    })
    .hide(axis="index")
)


     ZHMolContact | Top-K precision Evaluation Summary      
ZHMolContact | Evaluation Summary (N = 69)


Top-K,Mean Precision
top1,0.5942
top2,0.6232
top3,0.6039
top4,0.6087
top5,0.6058
top6,0.5966
top7,0.5983
top8,0.6051
top9,0.6039
top10,0.6058


In [10]:
# ==================== AUROC Evaluation Configuration ====================
os.makedirs(result_dir, exist_ok=True)

auroc_save_path = os.path.join(
    result_dir, "ZHMolContact_auroc_results.csv"
)

# Initialize evaluation results
auroc_results = []

# ==================== Model Evaluation ====================
zhmolcontact.eval()

for data in tqdm(new_test_loader, desc="Evaluating AUROC"):

    # ==================== Prepare Input ====================
    data = pad_conmap_truth_lbl(data)
    data = rigid_transform_perturb_batch(data, seed=seed, oprate_trans=False)
    data = data.to(device)
    test_id = data.pdbname[0]

    # ==================== Model Prediction ====================
    # Reuse predictions saved by the precision pass instead of running the model twice.
    probability_file = os.path.join(pred_dir, f'{test_id}.csv')
    final_probs_2d = torch.as_tensor(
        pd.read_csv(probability_file, header=None).to_numpy(), device=device, dtype=torch.float32
    )

    # ==================== Prepare Ground-Truth Labels ====================
    lbl_target = data.conmap_truth_lbl_pad  # [B, H, W]
    res_lbl = lbl_target[0]

    # Exclude padding positions
    valid_mask = res_lbl != -100

    # Convert distance labels to binary contact labels
    y_true = (res_lbl[valid_mask] < 16).int().cpu().numpy()
    y_score = final_probs_2d[valid_mask].detach().cpu().numpy()

    # ==================== Compute AUROC ====================
    if len(np.unique(y_true)) == 2:
        auroc = roc_auc_score(y_true, y_score)
    else:
        auroc = np.nan

    # Store evaluation result
    auroc_results.append({
        "pdbcode": test_id,
        "auroc": auroc
    })

    print(f"{test_id:<20} | AUROC: {auroc:.4f}")

# ==================== Save Evaluation Results ====================
auroc_df = pd.DataFrame(auroc_results)

# Add sequential index
auroc_df.insert(0, "index", range(len(auroc_df)))

# Save AUROC results
auroc_df.to_csv(auroc_save_path, index=False)

Evaluating AUROC:   0%|          | 0/69 [00:00<?, ?it/s]

8IYQ_O_A             | AUROC: 0.7590
8JOZ_B_A             | AUROC: 0.9872
9C69_C_A             | AUROC: 0.9210
9C68_C_A             | AUROC: 0.9054
9F37_V_A             | AUROC: 0.9940
9F37_V_B             | AUROC: 0.9956
8R0S_C_A             | AUROC: 0.9178
8PNP_V_A             | AUROC: 0.9936
8PNP_V_B             | AUROC: 0.9904
8PNP_V_C             | AUROC: 0.9905
8GXB_A_C             | AUROC: 0.9965
8XC1_C_A             | AUROC: 0.8715
8XC1_D_A             | AUROC: 0.9235
8BH9_B_A             | AUROC: 0.9855
8DP3_R_H             | AUROC: 0.9995
8DP3_R_L             | AUROC: 0.9952
8Q42_X_A             | AUROC: 0.9329
8TG4_C_A             | AUROC: 0.6222
8THQ_R_A             | AUROC: 0.9781
8PSO_S_A             | AUROC: 0.7979
8PSO_S_B             | AUROC: 0.7367
7YEW_B_A             | AUROC: 0.9788
7YEX_B_A             | AUROC: 0.9835
7YEY_B_A             | AUROC: 0.7764
8Z85_D_A             | AUROC: 0.7247
8Z85_E_A             | AUROC: 0.5798
8Z85_D_B             | AUROC: 0.7542
8

In [11]:
# ==================== Summary Statistics ====================
mean_auroc = auroc_df["auroc"].mean()
std_auroc = auroc_df["auroc"].std()
median_auroc = auroc_df["auroc"].median()

summary_df = pd.DataFrame({
    "Metric": ["Mean AUROC", "Std AUROC", "Median AUROC"],
    "Value": [mean_auroc, std_auroc, median_auroc]
})

print(f"\n{'=' * 60}")
print(f"{'ZHMolContact | AUROC Evaluation Summary':^60}")
print(f"{'=' * 60}")
print(f"Total complexes: {len(auroc_df)}")
print(f"Valid AUROC values: {auroc_df['auroc'].notna().sum()}")
print(f"Results saved to: {auroc_save_path}")

display(
    summary_df.style
    .format({"Value": "{:.4f}"})
    .hide(axis="index")
)


          ZHMolContact | AUROC Evaluation Summary           
Total complexes: 69
Valid AUROC values: 69
Results saved to: result/FoldBench_set/ZHMolContact_auroc_results.csv


Metric,Value
Mean AUROC,0.8799
Std AUROC,0.1291
Median AUROC,0.9210
